# Smart Lender – Loan Eligibility Prediction System

This notebook demonstrates the end-to-end Machine Learning pipeline for predicting loan eligibility. The goal is to build a classification model to automate loan approvals based on applicant parameters.

## Objectives:
1. **Exploratory Data Analysis (EDA)**: Understand demographics, correlations, and distributions.
2. **Data Preprocessing**: Impute missing values, encode categorical variables, scale numerical inputs, and apply SMOTE to resolve class imbalances.
3. **Model Training**: Evaluate Decision Tree, Random Forest, KNN, and XGBoost Classifiers.
4. **Model Serialization**: Save the best-performing model and transformers for Flask deployment.

In [ ]:
# Import Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import pickle

%matplotlib inline
sns.set_theme(style="whitegrid")


In [ ]:
# 1. Load Dataset
data_path = '../dataset/loan_data.csv'
if not os.path.exists(data_path):
    # Fallback to local execution folder if run from inside notebook directory
    data_path = 'dataset/loan_data.csv'

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# 2. Exploratory Data Analysis (EDA)
print("Missing Values:")
print(df.isnull().sum())

print("\nClass Distribution of Loan_Status:")
print(df['Loan_Status'].value_counts(normalize=True))

In [ ]:
# 3. Preprocessing: Imputation, Encoding and Scaling
df_prep = df.copy().drop(columns=['Loan_ID'])

# Categorical & Discrete columns
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

# Impute categorical & Credit_History using mode
for col in cat_cols + ['Credit_History']:
    df_prep[col] = df_prep[col].fillna(df_prep[col].mode()[0])

# Impute numerical using mean
for col in num_cols:
    df_prep[col] = df_prep[col].fillna(df_prep[col].mean())

# Label Encode categorical columns
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_prep[col] = le.fit_transform(df_prep[col].astype(str))
    encoders[col] = le

# Map Loan_Status to binary
df_prep['Loan_Status'] = df_prep['Loan_Status'].map({'Y': 1, 'N': 0})

# Prepare features and target
X = df_prep.drop(columns=['Loan_Status'])
y = df_prep['Loan_Status']

# Standardize numerical features
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

X.head()

In [ ]:
# 4. Split and Train Models (Handling Imbalance via SMOTE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE on training set only
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Define Models
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# Train and evaluate
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cv_score = cross_val_score(model, X_train_res, y_train_res, cv=5).mean()
    print(f"{name:15} | Accuracy: {acc*100:.2f}% | 5-Fold CV: {cv_score*100:.2f}%")